In [1]:
from pathlib import Path

from trait_prediction.main import (
    FeatureIndex,
    FeatureInput,
    PhenotypeIndex,
    PhenotypeInput,
    DataSet
)

In [2]:
def generate_phenotype_pinputs(phenotype_folder):
    pinputs = []
    for phenotype_file in phenotype_folder.glob("**/*.tsv"):
        name = phenotype_file.stem
        category = phenotype_file.parent.stem
        pindex = PhenotypeIndex(name=name, category=category)
        index_format_func = lambda x: x  # noqa
        pinput = PhenotypeInput(
            path=phenotype_file, pindex=pindex, index_format_func=index_format_func
        )
        pinputs.append(pinput)
    return pinputs


def generate_feature_finputs(feature_folder):
    finputs = []
    for feature_file in feature_folder.glob("**/*.tsv"):
        name = feature_file.stem
        ftype = "binary"
        dtype = "uint8"
        findex = FeatureIndex(name=name, ftype=ftype, dtype=dtype)
        index_format_func = lambda x: x  # noqa
        finput = FeatureInput(
            path=feature_file, findex=findex, index_format_func=index_format_func
        )
        finputs.append(finput)
    return finputs

In [3]:
def make_dataset(dataset):
    pinputs = generate_phenotype_pinputs(
        Path(f"../data/processed/phenotypes/{dataset}")
    )
    finputs = generate_feature_finputs(
        Path(f"../data/processed/features_reduced/{dataset}")
    )
    dataset = DataSet.read_data(pinputs, finputs)
    return dataset

In [4]:
atleaf_dataset = make_dataset("atleaf")
atleaf_dataset

DataSet(phenotype_set=PhenotypeSet (n=44), feature_set=FeatureSet (n=14))

In [5]:
def get_data(dataset):
    data = []
    for feature_raw in dataset.feature_set:
        for phenotype_raw in dataset.phenotype_set:
            phenotype, feature = dataset.get_data(
                phenotype_raw.pindex, feature_raw.findex
            )
            data.append((phenotype, feature))
    return data


In [6]:
atleaf_data = get_data(atleaf_dataset)

In [7]:
atleaf_data[0]

(Phenotype (name=Xylose, category=atleaf, size=202),
 Feature (name=uniprot_trembl, type=binary, dtype=uint8, n_genomes=202, n_features=379))

In [8]:
import pandas as pd

In [9]:
def is_xdata_good(feature_data: pd.DataFrame) -> bool:
    """Check if the feature data is good for training.

    Parameters
    ----------
    feature_data : pd.DataFrame
        The data frame containing the feature data.

    Returns
    -------
    bool
        True if the data is good for training, otherwise False.
    """
    if feature_data.shape[0] <= 20:
        return False
    if feature_data.shape[1] <= 1:
        return False
    return True

def is_ydata_good(phenotype_data: pd.Series) -> bool:
    """Check if the phenotype data is good for training.

    Parameters
    ----------
    phenotype_data : pd.Series
        The series containing the phenotype data.

    Returns
    -------
    bool
        True if the data is good for training, otherwise False.
    """
    # Skip if phenotype has only one class
    if len(phenotype_data.unique()) == 1:
        return False
    if phenotype_data.shape[0] <= 20: return False
    if (
        phenotype_data.value_counts().min()
        <= 10
    ):
        return False
    return True



In [10]:
pindex = PhenotypeIndex(name="Arabinose", category="atleaf")
findex = FeatureIndex(name="kofam", ftype="binary", dtype="uint8")
phenotype, feature = atleaf_dataset.get_data(pindex, findex)
phenotype, feature

(Phenotype (name=Arabinose, category=atleaf, size=204),
 Feature (name=kofam, type=binary, dtype=uint8, n_genomes=204, n_features=3925))

In [11]:
phenotype_data = phenotype.phenotype_data
feature_data = feature.feature_data
is_xdata_good(feature_data), is_ydata_good(phenotype_data)

(True, True)

In [12]:
from trait_prediction.main import Feature

In [13]:
feature_data, low_variance_features = Feature.remove_features_with_low_variance(feature_data, threshold=0.01)

In [14]:
feature_data

,K09681,K11921,K00241,K22310,K23107,K01179,K06400,K07492,K08166,K02529,...,K12276,K10924,K12287,K20966,K20922,K26913,K21712,K18678,K21252,K12954
genomeID,,,,,,,,,,,,,,,,,,,,,
GCF_001421155.1,0,0,1,0,0,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
GCF_001421165.1,0,0,1,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
GCF_001421245.1,0,0,1,0,0,1,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0
GCF_001421275.1,0,0,1,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
GCF_001421285.1,0,0,1,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCF_920984695.1,0,0,1,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
GCF_920984705.1,0,0,1,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
GCF_920984725.1,0,0,1,0,1,0,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0


In [15]:
low_variance_features

['K18307',
 'K18306',
 'K01635',
 'K15878',
 'K06212',
 'K19661',
 'K22072',
 'K22895',
 'K21563',
 'K00376',
 'K09700',
 'K18430']

In [16]:
feature_data, corr_group_dict = Feature.remove_features_with_high_correlation(feature_data, threshold=0.95, parallel=False)

In [17]:
corr_group_dict

{'K11747': ['K00992'],
 'K00992': ['K11747'],
 'K05594': ['K05812'],
 'K05812': ['K05594'],
 'K01577': ['K07230'],
 'K07230': ['K01577']}